In [2]:
import requests
import pandas as pd

In [3]:
import mysql.connector
from mysql.connector import Error
import numpy as np

In [3]:
import os

In [4]:
import requests
import pandas as pd

def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Janis Joplin", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,13840,Janis Joplin,Piece of My Heart,The Essential Janis Joplin,colaboracion,2003,Rock,152
1,1658,Janis Joplin,Me and Bobby McGee,Pearl (Legacy Edition),album,1971,Rock,152
2,1658,Janis Joplin,Cry Baby,Pearl (Legacy Edition),album,1971,Rock,152
3,1658,Janis Joplin,Kozmic Blues,I Got Dem Ol' Kozmic Blues Again Mama!,album,1969,Rock,152
4,13840,Janis Joplin,Call On Me,Box Of Pearls,colaboracion,1999,Rock,152
5,13840,Janis Joplin,Summertime,The Essential Janis Joplin,colaboracion,2003,Rock,152
6,1658,Janis Joplin,Little Girl Blue,I Got Dem Ol' Kozmic Blues Again Mama!,album,1969,Rock,152
7,1658,Janis Joplin,Move Over,Pearl,album,1971,Pop,132
8,13840,Janis Joplin,Ball and Chain,Iconic Performances from the Monterey Internat...,colaboracion,2017,Rock,152
9,1658,Janis Joplin,Maybe,Janis: Little Girl Blue (Original Motion Pictu...,album,2016,Rock,152


In [5]:
df_artista.to_csv("janis.csv", index=False)

In [4]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Celia Cruz", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,5024,Celia Cruz,La Vida Es Un Carnaval,La Vida Es Un Carnaval (Baile Total),album,2017,Latino,197
1,5024,Celia Cruz,La Dicha Mía (Remastered 2026),"Celia, Johnny and Pete (Remastered 2026)",colaboracion,2026,Latino,197
2,5024,Celia Cruz,Oye Cómo Va,"Fania Records: The 60's, Vol. Three",colaboracion,2019,Latino,197
3,2789,Celia Cruz,El Ultimo Adiós Varios Artistas Version,El Ultimo Adiós,colaboracion,2001,Pop,132
4,5024,Celia Cruz,La Negra Tiene Tumbao,La Reina Y Sus Amigos,album,2009,Latino,197
5,84998,Celia Cruz,Quimbara,"Salsa Party, Vol. 2",colaboracion,2019,Latino,197
6,5024,Celia Cruz,Yo Viviré (I Will Survive),Siempre Viviré,album,2000,Salsa,67
7,84998,Celia Cruz,Canto A La Habana,Fania Classics: Celia Cruz & Johnny Pacheco,colaboracion,2019,Latino,197
8,5678,Celia Cruz,A lo loco (con Celia Cruz),Grandes Exitos?,colaboracion,2003,Latino,197
9,5024,Celia Cruz,Rie y Llora,Soy Mujer,album,2014,Latino,197


In [5]:
df_artista.to_csv("celia.csv", index=False)

In [6]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Aretha Franklin", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,2059,Aretha Franklin,Respect,I Never Loved a Man the Way I Love You,album,1995,Pop,132
1,2059,Aretha Franklin,I Say a Little Prayer,Soul Queen,album,2007,Pop,132
2,2059,Aretha Franklin,Think,Aretha Now,album,1993,Pop,132
3,2059,Aretha Franklin,(You Make Me Feel Like) A Natural Woman,Lady Soul (With Bonus Selections),album,1987,Pop,132
4,2708,Aretha Franklin,Think,The Blues Brothers Original Motion Picture Sou...,colaboracion,2011,Pop,132
5,2059,Aretha Franklin,Today I Sing the Blues (Remastered),The Very Best of Blues : 50 Unforgettable Trac...,album,2013,Rock,152
6,264516192,Aretha Franklin,I Knew You Were Waiting (For Me),Aretha (Expanded Edition),colaboracion,1986,R&B,165
7,2059,Aretha Franklin,I Never Loved a Man (The Way I Love You),I Never Loved a Man the Way I Love You,album,1995,Pop,132
8,2059,Aretha Franklin,Respect,Soul Queen,album,2007,Pop,132
9,2059,Aretha Franklin,Chain of Fools,Soul Queen,album,2007,Pop,132


In [7]:
df_artista.to_csv("aretha.csv", index=False)

In [8]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Lola Flores", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,93523,Lola Flores,Limosna De Amores,Canta,album,2025,Latino,197
1,93523,Lola Flores,Capri C'est Fini,"Lola Flores: Sus Mejores Exitos, Vol. 1",album,2006,NaN,-1
2,93523,Lola Flores,A Tu Vera (Remastered),Grabaciones Completas (Remastered),album,2018,Flamenco,36
3,93523,Lola Flores,Como Me las Maravillaría Yo,"Todos Sus Exitos, Vol. 1",album,1994,Latino,197
4,93523,Lola Flores,Historia de un Amor (Remastered),Grabaciones Completas (Remastered),album,2018,Flamenco,36
5,93523,Lola Flores,Adoro,La Légende...,album,2011,NaN,-1
6,93523,Lola Flores,El Meneito,Mitos de la Musica Española : Lola Flores y An...,colaboracion,2000,NaN,-1
7,93523,Lola Flores,Que Me Coma el Tigre (Remastered),Grabaciones Completas (Remastered),album,2018,Flamenco,36
8,93523,Lola Flores,Díme,"Antonio Gonzalez ""El Pescailla""",colaboracion,1989,NaN,-1
9,93523,Lola Flores,"Ay, pena, penita, pena","Ay Pena, Penita, Pena",single,2007,Pop,132


In [9]:
df_artista.to_csv("lola.csv", index=False)

In [10]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Nina Simone", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,744,Nina Simone,Feeling Good,Feeling Good: The Very Best Of Nina Simone,album,2019,Jazz,129
1,744,Nina Simone,I Put A Spell On You,I Put A Spell On You,album,2018,Jazz,129
2,744,Nina Simone,One September Day,I Put A Spell On You,album,2018,Jazz,129
3,744,Nina Simone,Here Comes the Sun,The Very Best Of Nina Simone 1967-1972 - Sugar...,album,1998,Jazz,129
4,744,Nina Simone,Sinnerman,Feeling Good: The Very Best Of Nina Simone,album,2019,Jazz,129
5,744,Nina Simone,Sinnerman (Sofi Tukker Remix),Sinnerman (Sofi Tukker Remix),colaboracion,2021,Jazz,129
6,744,Nina Simone,Sinnerman (Felix Da Housecat's Heavenly House ...,Verve Remixed 2,colaboracion,2007,Jazz,129
7,744,Nina Simone,My Baby Just Cares for Me,My Baby Just Cares for Me,album,2009,Jazz,129
8,744,Nina Simone,Take Care Of Business (Solomun Mix),Take Care Of Business (Solomun Mix),colaboracion,2026,Electro,106
9,744,Nina Simone,Ain't Got No - I Got Life (From the Musical Pr...,The Greatest Hits,album,2003,Pop,132


In [11]:
df_artista.to_csv("nina.csv", index=False)

In [12]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Sylvie Vartan", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,11440,Sylvie Vartan,La Maritza,La Maritza,album,2026,Pop,132
1,11440,Sylvie Vartan,L'amour c'est comme une cigarette,Le meilleur des années RCA,album,2026,Pop,132
2,11440,Sylvie Vartan,La plus belle pour aller danser,La plus belle pour aller danser,ep,2026,Pop,132
3,11440,Sylvie Vartan,La plus belle pour aller danser,Sylvie Vartan,album,1995,Pop,132
4,13314683,Sylvie Vartan,Baby c'est vous,Rivages,colaboracion,2024,Electro,106
5,11440,Sylvie Vartan,Irrésistiblement,Les années RCA (Vol. 3),album,2026,Pop,132
6,11440,Sylvie Vartan,Comme un garçon,Le meilleur des années RCA,album,2026,Pop,132
7,11440,Sylvie Vartan,Ne t'en va pas,Twiste et chante,album,2026,Pop,132
8,11440,Sylvie Vartan,La drôle de fin (Last Tango),Qu'est ce qui fait pleurer les blondes?,album,2026,Pop,132
9,11440,Sylvie Vartan,Qu'est ce qui fait pleurer les blondes (Ride t...,Triple Best Of,album,2026,Pop,132


In [13]:
df_artista.to_csv("silvie.csv", index=False)

In [14]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Patty Pravo", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,58615,Patty Pravo,La bambola,Aristocratica,album,1998,Pop,132
1,58615,Patty Pravo,...dimmi che non vuoi morire,Bye bye patty,album,1997,Pop,132
2,58615,Patty Pravo,La bambola,Patty Pravo,album,2000,Pop,132
3,58615,Patty Pravo,Lettera a Gianni,Patty Pravo - Rarities 1967,album,2017,Pop,132
4,58615,Patty Pravo,Pazza idea,Anni Settanta,album,2007,Pop,132
5,58615,Patty Pravo,La Spada Nel Cuore,Mi ritorni in mente...i successi degli anni '70,album,2007,Pop,132
6,58615,Patty Pravo,Il paradiso,Patty Pravo - I Miti,album,1999,Pop,132
7,58615,Patty Pravo,Tutt'al più,Patty Pravo - I Miti,album,1999,Pop,132
8,58615,Patty Pravo,I giardini di Kensington,Patty Pravo - I Miti,album,1999,Pop,132
9,58615,Patty Pravo,Morire tra le viole,Patty Pravo - I Miti,album,1999,Pop,132


In [15]:
df_artista.to_csv("patty.csv", index=False)

In [16]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Rita Pavone", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,77224,Rita Pavone,Cuore,31 Grandi Successi,album,2016,Pop,132
1,77224,Rita Pavone,Viva la pappa col pomodoro,Los Grandes Exitos de Rita Pavone,album,1998,Pop,132
2,77224,Rita Pavone,Amore twist,Labor day (Italian music party),album,2016,Pop,132
3,77224,Rita Pavone,Que Me Importa el Mundo,Rita Pavone canta en Espanol (Singles Collection),album,2000,Pop,132
4,77224,Rita Pavone,Il ballo del mattone,Rita Pavone (Gli Indimenticabili),album,2026,Pop,132
5,77224,Rita Pavone,Come te non c'e' nessuno,The Best of Rita Pavone,album,2014,NaN,-1
6,77224,Rita Pavone,Corazón,Sus Exitos en Español,album,1995,Latino,197
7,77224,Rita Pavone,Zucchero,Nostalgia Italiana - 1969,album,1996,Pop,132
8,77224,Rita Pavone,Viva la pappa col pomodoro,31 Grandi Successi,album,2016,Pop,132
9,77224,Rita Pavone,Che m´importa del mondo,31 Grandi Successi,album,2016,Pop,132


In [17]:
df_artista.to_csv("rita.csv", index=False)

In [18]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": 85766522,
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Karina", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,85766522,Karina,Otoño Porteño,"Musica Porteña, Il Tango di Astor Piazzolla",colaboracion,2012,Tango,73
1,85766522,Karina,Jacinto Chiclana,"Musica Porteña, Il Tango di Astor Piazzolla",colaboracion,2012,Tango,73
2,85766522,Karina,Milonga de la Anunciaciòn,"Musica Porteña, Il Tango di Astor Piazzolla",colaboracion,2012,Tango,73
3,85766522,Karina,Crazy in Love (Electro Swing Version),The Best of Swing Republic,colaboracion,2018,Electro,106
4,85766522,Karina,El baúl de los recuerdos (2015 Remastered Vers...,Éxitos de 1970. Artistas Originales (Remastere...,album,2015,Pop,132
5,85766522,Karina,Satisfeito,Universo Ao Meu Redor,album,2006,Música Brasileña,75
6,85766522,Karina,SINVERGÜENZA - con Angela Torres,SINVERGÜENZA - con Angela Torres,colaboracion,2023,Latino,197
7,85766522,Karina,Rebota,Rebota,colaboracion,2025,Latino,197
8,85766522,Karina,Sé Cómo Duele,Serie 32 Grandes Éxitos,album,2022,Latino,197
9,85766522,Karina,LADRON,LADRON,colaboracion,2025,Latino,197


In [19]:
df_artista.to_csv("karina.csv", index=False)

In [20]:
def conseguir_canciones(artist_name, limit=50):     # Función para conseguir 50 canciones de la artista que quieras
    url = "https://api.deezer.com/search"           # URL base de la API para buscar
    params = {                                      # Para insertar los parámetros de la búsqueda
        "q": f'artist:"{artist_name}"',             # La API busca artista por q
        "limit": limit                              # Limit sirve para que saque solo 50
    }
    try:                                            # Bloque try/excep para llamar a la API y que nos dé el resultado
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("Error en la llamada a Deezer:", response.status_code)    # Si falla lo va a informar
            return pd.DataFrame([])                                         # Y también nos va a devolver el DF vacío
        data = response.json()                                              # Si no falla da la respuesta en json
    except Error as e:
        print("Error en la llamada a Deezer:", e)                           # Para que nos diga exactamente cuál es el error
        return pd.DataFrame([])

    canciones = data.get("data", [])                # Obtiene la lista de canciones
    lista_canciones = []                            # Lista donde se van a acumular las filas para el DF final

    for cancion in canciones:                       # Bucle para que itere con todas las canciones y nos vaya dando los datos
        track_id = cancion["id"]                    # Nos da el id de la canción
        track = requests.get(f"https://api.deezer.com/track/{track_id}").json()  # Pide todos los datos que estén en track
        album = requests.get(f"https://api.deezer.com/album/{track['album']['id']}").json() # Pide todos los datos que estén en album

        genre_id = album.get("genre_id")   # Extrae el ID del género de album
        genre_name = None                  # Creamos la variable "vacía"
        if album.get("genres") and album["genres"].get("data"):   # Le decimoa que en el caso de que genres exista en album
            genre_name = album["genres"]["data"][0].get("name")   # Entonces nos diga el primer género que aparezca (por si hay varios)

        tracks_type = "colaboracion" if len(track.get("contributors", [])) > 1 else album.get("record_type", "track")
        # Buscamos colaboraciones. Para ello, si en contributor aparece más de 1, entonces se trata de una colaboración
        # Si no, le decimos que nos ponga lo que aparezca en record_type que es single o album, y si no que ponga track
        release_year = track.get("release_date", "")[:4] if track.get("release_date") else None
        # Obtiene el año de lanzamiento solo si hay 4 cifras, si no pondrá None 
        lista_canciones.append({                   # Creamos un diccionario para añadir todos los datos que necesitamos
            "id_artista": track["artist"]["id"],
            "nombre_artista": artist_name,
            "titulo_cancion": track["title"],
            "titulo_album": track["album"]["title"],
            "tipo": tracks_type,
            "año_lanzamiento": release_year,
            "genero": genre_name,
            "id_genero": genre_id,
        })

    return pd.DataFrame(lista_canciones)        # Hacemos que nos devuelta el resultado en un DF

df_artista = conseguir_canciones("Massiel", limit=50)
df_artista

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,54310142,Massiel,"Me Dijeron (feat. DVG, Magdiel CRZ, Risen, Lir...","Me Dijeron (feat. DVG, Magdiel CRZ, Risen, Lir...",colaboracion,2023,Rap/Hip Hop,116
1,105903,Massiel,Sur Fond De Marseillaise,Demain c'est toi,album,2005,Rap/Hip Hop,116
2,10396026,Massiel,Un Mundo Feliz,Manual de Belleza,colaboracion,2026,Pop latino,138
3,65759,Massiel,"Berlioz: Hymne des Marseillais, H 51a",Plácido Domingo - Artist Portrait 2007,colaboracion,2004,Clásica,98
4,129722,Massiel,Eres,Lo Mejor de Massiel,album,2013,Pop,132
5,129722,Massiel,El amor,Lo Mejor de Massiel,album,2013,Pop,132
6,129722,Massiel,"La, La, La",Lo Mejor de Massiel,album,2015,Pop,132
7,129722,Massiel,El amor (en directo),Massiel En Des. Concierto,album,2014,Pop,132
8,129722,Massiel,Brindaremos por él,Lo Mejor de Massiel,album,2013,Pop,132
9,129722,Massiel,El Noa-Noa,Lo Mejor de Massiel,album,2013,Pop,132


In [21]:
df_artista.to_csv("massiel.csv", index=False)

In [5]:
LASTFM_API_KEY = "88d4c2ca98c2eacf39cbd94fdd2a63ba"

In [ ]:
def obtener_info_lastfm(artist_name):
    
    LASTFM_BASE_URL = "https://ws.audioscrobbler.com/2.0/" #endpoint

    params = {                                             #parámetros para buscar
        "method": "artist.getInfo",
        "artist": artist_name,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1
    }

    data = requests.get(
        LASTFM_BASE_URL,
        params=params
    ).json()

    if "artist" not in data:
        return {
            "nombre_artista": artist_name,
            "biografía": None,
            "listeners": None,
            "playcount": None
        }

    artist = data["artist"]

    return {
        "nombre_artista": artist_name,
        "biografía": artist.get("bio", {}).get("summary"),
        "listeners": artist.get("stats", {}).get("listeners"),
        "playcount": artist.get("stats", {}).get("playcount")
    }
artistas = ["Janis Joplin", "Celia Cruz", "Aretha Franklin", "Lola Flores", "Nina Simone", "Silvie Vartan", "Patty Pravo", "Rita Pavone", "Karina", "Massiel"]

# Obtener info de cada artista
lista_info = [obtener_info_lastfm(artista) for artista in artistas]

# Convertir a DataFrame
df_info = pd.DataFrame(lista_info)
df_info

,artist_name,biography,listeners,playcount
0,Janis Joplin,Janis Joplin (born 19 January 1943 in Port Art...,1937889,34092778
1,Celia Cruz,Celia Cruz (born as Úrsula Hilaria Celia de la...,587427,5043428
2,Aretha Franklin,"Aretha Franklin (March 25, 1942 - August 16, 2...",2932315,53111402
3,Lola Flores,"Lola Flores, ""La Faraona"" born January 21, 192...",43166,351079
4,Nina Simone,Eunice Kathleen Waymon (21 February 1933 – 21 ...,2995908,75195380
5,Silvie Vartan,"Sylvie Vartan (born August 15, 1944 in Iskretz...",206481,1944467
6,Patty Pravo,"Patty Pravo (born Nicoletta Strambelli, 9 Apri...",113871,1015532
7,Rita Pavone,"Rita Pavone (born August 23, 1945 - Turin) is ...",77769,554641
8,Karina,There are multiple artists with this name:\n\n...,140669,1869320
9,Massiel,Massiel (real name Maria de los Angeles Santam...,39865,234507


In [9]:
df_info.to_csv("informacion_artist.csv", index=False)

In [10]:
df_info.to_csv("artist_similares", index=False)

In [ ]:
import glob

archivos = glob.glob("*.csv")
artista_completo_Teresa = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)
artista_completo_Teresa.to_csv("artista_completo_Teresa.csv", index=False, encoding="utf-8")

In [6]:
import requests
import pandas as pd

LASTFM_BASE_URL = "https://ws.audioscrobbler.com/2.0/"

def obtener_info_lastfm(artist_name):
    params = {
        "method": "artist.getInfo",
        "artist": artist_name,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1
    }

    data = requests.get(LASTFM_BASE_URL, params=params).json()

    if "artist" not in data:
        return {
            "nombre_artista": artist_name,
            "biografía": None,
            "listeners": None,
            "playcount": None
        }

    artist = data["artist"]

    return {
        "nombre_artista": artist_name,
        "biografía": artist.get("bio", {}).get("summary"),
        "listeners": artist.get("stats", {}).get("listeners"),
        "playcount": artist.get("stats", {}).get("playcount")
    }


def obtener_artistas_similares(artist_name):
    params = {
        "method": "artist.getSimilar",
        "artist": artist_name,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1,
        "limit": 10
    }

    data = requests.get(LASTFM_BASE_URL, params=params).json()

    if "similarartists" not in data:
        return []

    return [
        artista["name"]
        for artista in data["similarartists"]["artist"]
    ]


artistas = ["Janis Joplin", "Celia Cruz", "Aretha Franklin", "Lola Flores", "Nina Simone",
            "Silvie Vartan", "Patty Pravo", "Rita Pavone", "Karina", "Massiel"]

lista_info = [obtener_info_lastfm(artista) for artista in artistas]
lista_similares = [obtener_artistas_similares(artista) for artista in artistas]


df_info = pd.DataFrame(lista_info)

df_similares = pd.DataFrame({
    "nombre_artista": artistas,
    "artistas_similares": lista_similares
})


df_completo = pd.merge(df_info, df_similares, on="nombre_artista", how="left")
df_completo

,nombre_artista,biografía,listeners,playcount,artistas_similares
0,Janis Joplin,Janis Joplin (born 19 January 1943 in Port Art...,1937889,34092778,"[Big Brother & The Holding Company, Jimi Hendr..."
1,Celia Cruz,Celia Cruz (born as Úrsula Hilaria Celia de la...,587427,5043428,"[Johnny Pacheco, Marc Anthony, Oscar D'León, R..."
2,Aretha Franklin,"Aretha Franklin (March 25, 1942 - August 16, 2...",2932315,53111402,"[Etta James, Gladys Knight & The Pips, Roberta..."
3,Lola Flores,"Lola Flores, ""La Faraona"" born January 21, 192...",43166,351079,"[Rocío Jurado, Camarón de la Isla, Isabel Pant..."
4,Nina Simone,Eunice Kathleen Waymon (21 February 1933 – 21 ...,2995908,75195380,"[Billie Holiday, Aretha Franklin, Etta James, ..."
5,Silvie Vartan,"Sylvie Vartan (born August 15, 1944 in Iskretz...",206481,1944467,"[Sheila, Françoise Hardy, Brigitte Bardot, Gil..."
6,Patty Pravo,"Patty Pravo (born Nicoletta Strambelli, 9 Apri...",113871,1015532,"[Anna Oxa, Mina, Loredana Bertè, Marcella Bell..."
7,Rita Pavone,"Rita Pavone (born August 23, 1945 - Turin) is ...",77769,554641,"[Edoardo Vianello, Little Tony, Peppino Di Cap..."
8,Karina,There are multiple artists with this name:\n\n...,140669,1869320,"[Cazzu, NINGNING, Emilia, Ha*Ash, Amanda Migue..."
9,Massiel,Massiel (real name Maria de los Angeles Santam...,39865,234507,"[Mari Trini, Rocío Jurado, Paloma San Basilio,..."


In [7]:
df_completo.to_csv("last_fm.csv", index=False)

In [23]:
import pandas as pd
import glob

ruta = "./concatenar/"
archivos = [f for f in glob.glob(ruta + "*.csv") if "artista_completo" not in f]
print("Archivos encontrados:", archivos)

artista_completo = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)
artista_completo.to_csv(ruta + "artista_completo.csv", index=False, encoding="utf-8")

print(f"Filas totales: {len(artista_completo)}")

Archivos encontrados: ['./concatenar\\aretha.csv', './concatenar\\celia.csv', './concatenar\\janis.csv', './concatenar\\karina.csv', './concatenar\\lola.csv', './concatenar\\massiel.csv', './concatenar\\nina.csv', './concatenar\\patty.csv', './concatenar\\rita.csv', './concatenar\\silvie.csv']
Filas totales: 454


In [24]:
print(archivos, "\n")

['./concatenar\\aretha.csv', './concatenar\\celia.csv', './concatenar\\janis.csv', './concatenar\\karina.csv', './concatenar\\lola.csv', './concatenar\\massiel.csv', './concatenar\\nina.csv', './concatenar\\patty.csv', './concatenar\\rita.csv', './concatenar\\silvie.csv'] 



In [5]:
artista_completo

,id_artista,nombre_artista,titulo_cancion,titulo_album,tipo,año_lanzamiento,genero,id_genero
0,744,Nina Simone,I Put A Spell On You,I Put A Spell On You,album,2018,Jazz,129
1,744,Nina Simone,Feeling Good,Feeling Good: The Very Best Of Nina Simone,album,2019,Jazz,129
2,744,Nina Simone,Ain't Got No - I Got Life (From the Musical Pr...,The Greatest Hits,album,2003,Pop,132
3,744,Nina Simone,Sinnerman,Feeling Good: The Very Best Of Nina Simone,album,2019,Jazz,129
4,744,Nina Simone,My Baby Just Cares for Me,My Baby Just Cares for Me,album,2009,Jazz,129
...,...,...,...,...,...,...,...,...
485,129722,Massiel,Todo lo que cambié por ti,iCollection,album,2016,Pop,132
486,129722,Massiel,Voy a empezar de nuevo,iCollection,album,2016,Pop,132
487,129722,Massiel,Miradas de amor,iCollection,album,2016,Pop,132
488,129722,Massiel,Tiempos difíciles,iCollection,album,2016,Pop,132


In [7]:
artista_completo["nombre_artista"].unique()

<StringArray>
[                                   'Nina Simone',
                                     'Celia Cruz',
                                   'Ricky Martin',
                                 'Johnny Pacheco',
       'Celia Cruz, Rene Hernandez Y Su Orquesta',
                                 'Jarabe de Palo',
                                 'Gloria Estefan',
                            'La Sonora Matancera',
                                  'Gente De Zona',
                                    'Tony Succar',
                                   'Willie Colón',
              'Big Brother & The Holding Company',
                                   'Janis Joplin',
                    'Paul Butterfield Blues Band',
 'Big Brother & The Holding Company;Janis Joplin',
                'Various Artists - Kobra Records',
                                        'Señor F',
                                 'Rodrigo Cuevas',
                                     'Marshmello',
                 

In [1]:
import os

ruta = "C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar/"

archivos = os.listdir(ruta)
print(f"Tienes {len(archivos)} archivos:")
for f in archivos:
    print(" -", f)

Tienes 10 archivos:
 - aretha.csv
 - celia.csv
 - janis.csv
 - karina.csv
 - lola.csv
 - massiel.csv
 - nina.csv
 - patty.csv
 - rita.csv
 - silvie.csv


In [2]:
import pandas as pd
import glob

ruta = "C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar/"

archivos = [f for f in glob.glob(ruta + "*.csv") if "artista_completo" not in f]
print("Archivos encontrados:", archivos)

artista_completo = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)
artista_completo.to_csv(ruta + "artista_completo.csv", index=False, encoding="utf-8")

print(f"Filas totales: {len(artista_completo)}")

Archivos encontrados: ['C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\aretha.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\celia.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\janis.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\karina.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\lola.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\massiel.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\nina.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\patty.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\rita.csv', 'C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar\\silvie.csv']
Filas totales: 498


In [12]:
## Conexión a Mysql
def conexion_mysql():
    try:
        conexion = mysql.connector.connect(
             host = '127.0.0.1',
             user = 'root',
             password = 'guitarra'
        )
        print("Conexion exitosa")
        return conexion
    except Exception as e:
        print(f"Error al conectar: {e}")
        
conexion = conexion_mysql()           

Conexion exitosa


In [13]:
cursor = conexion.cursor()

In [8]:
#Creación Base de Datos

try:
    cursor = conexion.cursor()
    query_crear_bbdd = """
    CREATE DATABASE IF NOT EXISTS musicstream
    CHARACTER SET utf8mb4
    COLLATE utf8mb4_unicode_ci
    """
    cursor.execute(query_crear_bbdd)
    print("Query existosa")
except Error as e:
    print(e)

Query existosa


In [9]:
cursor.execute("use musicstream")
query_crear_tabla = '''CREATE TABLE artistas (
                        id_artista INT PRIMARY KEY AUTO_INCREMENT, 
                        nombre_artista VARCHAR(50) NOT NULL
                        );'''
cursor.execute(query_crear_tabla)
print ("Tabla creada correctamente")

Tabla creada correctamente


In [17]:
cursor.execute("USE musicstream")

In [16]:
df_tabla_artistas = pd.read_csv(r"C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/git_teresa/proyecto-da-promo-67-modulo2-team-3/Deezer/artistas_completo_deezer.csv",
    encoding="utf-8"
)
df_tabla_artistas = df_tabla_artistas[["nombre_artista"]].drop_duplicates()
for index, fila in df_tabla_artistas.iterrows():
    query_insert = """
    INSERT INTO artistas (nombre_artista)
    VALUES (%s)
    """
    valores = (fila["nombre_artista"],)
    cursor.execute(query_insert, valores)
conexion.commit()
print("Datos insertados correctamente")

Datos insertados correctamente


In [ ]:
ruta = "C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/concatenar/"


In [5]:
print(os.listdir())

['.git', '.gitignore', 'Deezer', 'divas_inic.ipynb', 'LFM', 'README.md']


In [18]:
cursor.execute("use musicstream")
query_crear_tabla = """
CREATE TABLE canciones (
    id_cancion INT PRIMARY KEY AUTO_INCREMENT,
    id_artista INT NOT NULL,
    titulo_cancion VARCHAR(300) NOT NULL,
    titulo_album VARCHAR(300),
    tipo VARCHAR(100),
    año_lanzamiento INT,
    genero VARCHAR(100),
    id_genero FLOAT,
    FOREIGN KEY (id_artista)
    REFERENCES artistas(id_artista)
);
"""
cursor.execute(query_crear_tabla)
print("Tabla canciones creada correctamente")

Tabla canciones creada correctamente


In [19]:
df_tabla_canciones = pd.read_csv(r"C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/git_teresa/proyecto-da-promo-67-modulo2-team-3/Deezer/artistas_completo_deezer.csv",
    encoding="utf-8"
)
for index, fila in df_tabla_canciones.iterrows():
    query_id_artista = """
    SELECT id_artista
    FROM artistas
    WHERE nombre_artista = %s
    """
    cursor.execute(
        query_id_artista,
        (fila["nombre_artista"],)
    )
    resultado = cursor.fetchone()
    id_artista = resultado[0]
    query_insert = """
    INSERT INTO canciones (
        id_artista,
        titulo_cancion,
        titulo_album,
        tipo,
        año_lanzamiento,
        genero,
        id_genero
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """
    valores = (
        id_artista,
        fila["titulo_cancion"],
        fila["titulo_album"],
        fila["tipo"],
        fila["año_lanzamiento"],
        fila["genero"]
        if pd.notna(fila["genero"])
        else None,
        fila["id_genero"]
        if pd.notna(fila["id_genero"])
        else None
    )
    cursor.execute(query_insert, valores)
conexion.commit()
print("Datos insertados correctamente")

Datos insertados correctamente


In [20]:
query_crear_tabla = """
CREATE TABLE informacion_artistas (
    id_info INT PRIMARY KEY AUTO_INCREMENT,
    id_artista INT NOT NULL,
    nombre_artista VARCHAR(50),
    biografia TEXT,
    listeners BIGINT,
    playcount BIGINT,
    artistas_similares TEXT,
    FOREIGN KEY (id_artista)
    REFERENCES artistas(id_artista)
);
"""
cursor.execute(query_crear_tabla)
print("Tabla informacion_artistas creada correctamente")
 

Tabla informacion_artistas creada correctamente


In [23]:
df_lfm = pd.read_csv(r"C:/Users/Teresa/Desktop/Adalab/proyectos/proyectos_grupo/proyecto2/git_teresa/proyecto-da-promo-67-modulo2-team-3/LFM/artistas_completo_LFM.csv",
    encoding="utf-8"
)
for index, fila in df_lfm.iterrows():
    query_id_artista = """
    SELECT id_artista
    FROM artistas
    WHERE nombre_artista = %s
    """
    cursor.execute(
        query_id_artista,
        (fila["nombre_artista"],)
    )
    resultado = cursor.fetchone()
    id_artista = resultado[0]
    query_insert = """
    INSERT INTO informacion_artistas (
        id_artista,
        nombre_artista,
        biografia,
        listeners,
        playcount,
        artistas_similares
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    valores = (
        id_artista,
        fila["nombre_artista"],
        fila["biografia"],
        fila["listeners"],
        fila["playcount"],
        fila["artistas_similares"]
    )
    cursor.execute(query_insert, valores)
conexion.commit()
print("Datos insertados correctamente")

TypeError: 'NoneType' object is not subscriptable

In [ ]:
#Rocio
for index, fila in df_last.iterrows():
    query_id_artista = """
    SELECT id_artista
    FROM artistas
    WHERE nombre_artista = %s
    """
    cursor.execute(
        query_id_artista,
        (fila["nombre_artista"],)
    )
    resultado = cursor.fetchone()
    if resultado is None:
        print(f"Artista no encontrado: {fila['nombre_artista']}")
        continue

    id_artista = resultado[0]
    query_insert = """
    INSERT INTO informacion_artistas (
        id_artista,
        nombre_artista,
        biografia,
        listeners,
        playcount,
        artistas_similares
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    valores = (
        id_artista,
        fila["nombre_artista"],
        fila["biografia"],
        fila["listeners"],
        fila["playcount"],
        fila["artistas_similares"]
    )
    cursor.execute(query_insert, valores)
connection.commit()
print("Datos insertados correctamente")
 